# Non-linear static analysis with the Newton-Raphson method

The geometrically non-linear models, based on Donnell's equations, provide the
internal force vector $\boldsymbol{F}_{int}$, calculated with `Shell.calc_fint`,
and the tangent stiffness matrix

$$
\boldsymbol{K}_T = \boldsymbol{K}_C + \boldsymbol{K}_G
$$

calculated with `Shell.calc_kT`, or with `Shell.calc_kC` and
`Shell.calc_kG` using `NLgeom=True`. The tangent stiffness matrix is
the exact derivative of the internal force vector, such that the
Newton-Raphson iterations converge quadratically.

The example below applies a compression of about 37 times the first linear
buckling load in a single step, deep in the postbuckling regime, starting from
the linear solution, and verifies the order of convergence of the iterations:

In [1]:
import numpy as np
from structsolve import solve
from panels.shell import Shell

In [2]:
# definitions of tests/tests_shell/test_nonlinear.py used below
def scaling(vec, D):
    non_nulls = ~np.isclose(D, 0)
    vec = vec[non_nulls]
    D = D[non_nulls]
    return np.sqrt((vec*np.abs(1/D))@vec)

In [3]:
# from tests/tests_shell/test_nonlinear.py
def test_nonlinear():
    m = 6
    n = 6
    for model in [
            'plate_clpt_donnell',
            'cylshell_clpt_donnell',
            'cylshell_clpt_sanders',
            'plate_fsdt_donnell',
            'plate_tsdt_donnell',
                  ]:
        print('Testing model: %s' % model)
        s = Shell()

        s.model = model
        s.x1u = 0
        s.x1ur = 1
        s.x2u = 1
        s.x2ur = 1

        s.x1v = 1
        s.x1vr = 1
        s.x2v = 1
        s.x2vr = 1

        s.x1w = 0
        s.x1wr = 1
        s.x2w = 0
        s.x2wr = 1

        s.y1u = 1
        s.y1ur = 1
        s.y2u = 1
        s.y2ur = 1

        s.y1v = 0
        s.y1vr = 1
        s.y2v = 1
        s.y2vr = 1

        s.y1w = 0
        s.y1wr = 1
        s.y2w = 0
        s.y2wr = 1

        s.a = 4.
        s.b = 1.
        s.r = 1.e15
        s.stack = [90, 0, 90, 0]
        s.plyt = 1e-3*0.125
        E11 = 142.5e9
        E22 = E11/20
        G12 = G13 = G23 = 0.5*E22
        s.laminaprop = (E11, E22, 0.25, G12, G12, G12)
        s.m = m
        s.n = n

        load = 700
        Nxx = load/s.b
        # distributed axial load
        s.add_distr_load_fixed_x(s.a, funcx=lambda y: -Nxx, funcy=None, funcz=None, cte=False)
        # perturbation load
        s.add_point_load(s.a/2., s.b/2., 0, 0, 0.001, cte=True)

        #initial
        fext = s.calc_fext()
        c = solve(s.calc_kC(), fext, silent=True)
        plot_mesh, fields = s.uvw(c=c)
        print('  linear wmax', fields['w'].max())
        assert np.isclose(fields['w'].max(), 0.0026619, rtol=0.01)

        # solving using the full Newton-Raphson method, with the tangent
        # stiffness matrix updated at every iteration
        D = s.calc_kC().diagonal() # at beginning of load increment
        epsilon = 1.e-10
        errors = []
        while True:
            fint = s.calc_fint(c=c)
            Ri = fint - fext
            crisfield_test = scaling(Ri, D)/max(scaling(fext, D), scaling(fint, D))
            errors.append(crisfield_test)
            print('  iteration %d, crisfield_test %1.3e' % (len(errors) - 1, crisfield_test))
            if crisfield_test < epsilon:
                break
            if len(errors) > 30:
                raise RuntimeError('Not converged!')
            KT = s.calc_kT(c=c)
            c = c + solve(KT, -Ri, silent=True)

        plot_mesh, fields = s.uvw(c=c)
        print('  nonlinear wmax', fields['w'].max())
        assert np.isclose(fields['w'].max(), 0.004841, rtol=0.01)

        # quadratic convergence: once in the asymptotic range, the order
        # log(e_k+1)/log(e_k) approaches 2, whereas an inconsistent tangent
        # gives an order of 1 and needs hundreds of iterations
        assert len(errors) <= 15, 'too many iterations: %d' % len(errors)
        orders = [np.log(e1)/np.log(e0) for e0, e1 in zip(errors[:-1], errors[1:])
                  if 1.e-15 < e1 and e0 < 1.e-2]
        print('  convergence orders', orders)
        if 'fsdt' in model or 'tsdt' in model:
            #NOTE for this very thin plate, a/h = 8000, the residual of the
            #     shear deformation theories reaches its round-off floor, of
            #     the order of eps*(G/E)*(a/h)**2 ~ 1e-9, within the asymptotic
            #     range, where the order cannot be measured; their quadratic
            #     convergence is pinned by test_tangent_consistency.py
            continue
        assert len(orders) >= 2
        assert min(orders) > 1.6, 'convergence is not quadratic: %s' % orders

In [4]:
test_nonlinear()
print('test_nonlinear: passed')

Testing model: plate_clpt_donnell
  linear wmax 0.0026619005987101796
  iteration 0, crisfield_test 9.832e-01
  iteration 1, crisfield_test 6.213e-01
  iteration 2, crisfield_test 9.994e-01
  iteration 3, crisfield_test 2.552e-01
  iteration 4, crisfield_test 6.138e-01
  iteration 5, crisfield_test 5.155e-02
  iteration 6, crisfield_test 2.032e-02
  iteration 7, crisfield_test 1.330e-04
  iteration 8, crisfield_test 8.177e-08
  iteration 9, crisfield_test 4.688e-14


  nonlinear wmax 0.004841391648297454
  convergence orders [np.float64(1.8284184542938373), np.float64(1.8806661635947866)]
Testing model: cylshell_clpt_donnell
  linear wmax 0.0026619005987103257
  iteration 0, crisfield_test 9.832e-01
  iteration 1, crisfield_test 6.213e-01
  iteration 2, crisfield_test 9.994e-01
  iteration 3, crisfield_test 2.552e-01
  iteration 4, crisfield_test 6.138e-01
  iteration 5, crisfield_test 5.155e-02
  iteration 6, crisfield_test 2.032e-02
  iteration 7, crisfield_test 1.330e-04
  iteration 8, crisfield_test 8.177e-08
  iteration 9, crisfield_test 2.894e-14


  nonlinear wmax 0.004841391648297375
  convergence orders [np.float64(1.8284184271025288), np.float64(1.9102324778906343)]
Testing model: cylshell_clpt_sanders
  linear wmax 0.002661900598710281
  iteration 0, crisfield_test 9.832e-01
  iteration 1, crisfield_test 6.213e-01
  iteration 2, crisfield_test 9.994e-01
  iteration 3, crisfield_test 2.552e-01
  iteration 4, crisfield_test 6.138e-01
  iteration 5, crisfield_test 5.155e-02
  iteration 6, crisfield_test 2.032e-02


  iteration 7, crisfield_test 1.330e-04
  iteration 8, crisfield_test 8.177e-08
  iteration 9, crisfield_test 1.656e-14
  nonlinear wmax 0.0048413916482973685
  convergence orders [np.float64(1.8284184537713755), np.float64(1.9444130964980788)]
Testing model: plate_fsdt_donnell


  linear wmax 0.0026618880543280306
  iteration 0, crisfield_test 6.223e-01
  iteration 1, crisfield_test 6.561e-02
  iteration 2, crisfield_test 8.935e-01
  iteration 3, crisfield_test 6.348e-03
  iteration 4, crisfield_test 5.149e-02
  iteration 5, crisfield_test 2.085e-03
  iteration 6, crisfield_test 1.485e-03
  iteration 7, crisfield_test 6.588e-06
  iteration 8, crisfield_test 5.870e-09
  iteration 9, crisfield_test 6.801e-14


  nonlinear wmax 0.004841394020262595
  convergence orders [np.float64(0.5862727611234385), np.float64(1.0549156355592384), np.float64(1.831998031000207), np.float64(1.588672870295289), np.float64(1.5996708542864)]
Testing model: plate_tsdt_donnell


  linear wmax 0.0026618937913918026
  iteration 0, crisfield_test 6.224e-01
  iteration 1, crisfield_test 6.564e-02
  iteration 2, crisfield_test 8.936e-01
  iteration 3, crisfield_test 6.379e-03
  iteration 4, crisfield_test 5.153e-02
  iteration 5, crisfield_test 2.088e-03


  iteration 6, crisfield_test 1.486e-03
  iteration 7, crisfield_test 6.596e-06
  iteration 8, crisfield_test 5.875e-09
  iteration 9, crisfield_test 5.147e-14


  nonlinear wmax 0.004841393619411852
  convergence orders [np.float64(0.5867085858910455), np.float64(1.055093741395953), np.float64(1.8319753763221662), np.float64(1.5887643246689722), np.float64(1.6144485042902719)]
test_nonlinear: passed


The consistency between the tangent stiffness matrix and the internal force
vector is verified by a directional Taylor test, where the error of the linear
approximation of $\boldsymbol{F}_{int}$ must decrease quadratically with the step size:

In [5]:
from scipy.linalg import eigh
from panels import modelDB

In [6]:
# definitions of tests/tests_shell/test_tangent_consistency.py used below
MODELS = ['plate_clpt_donnell', 'cylshell_clpt_donnell',
          'cylshell_clpt_sanders', 'plate_fsdt_donnell', 'plate_tsdt_donnell',
          'cylshell_fsdt_donnell', 'cylshell_fsdt_sanders',
          'cylshell_tsdt_donnell', 'cylshell_tsdt_sanders', 'plate_clpt_gl',
          'plate_clpt_tl']


def make_shell(model):
    s = Shell()
    s.model = model
    s.a = 0.3
    s.b = 0.2
    s.r = 0.4 if model.startswith('cylshell') else None
    # unsymmetric and unbalanced, such that A16, A26 and B are non-zero
    s.stack = [30, -45, 0, 90]
    s.plyt = 0.125e-3
    E11 = 142.5e9
    E22 = E11/20
    G12 = 0.5*E22
    s.laminaprop = (E11, E22, 0.25, G12, G12, G12)
    s.m = 5
    s.n = 5
    s.nx = 8
    s.ny = 8
    s.Nxx = s.Nyy = s.Nxy = 0.
    s.Nxx_cte = s.Nyy_cte = s.Nxy_cte = 0.
    for edge in ('x1', 'x2', 'y1', 'y2'):
        for field in 'uvw':
            setattr(s, edge + field, 0. if field == 'w' else 1.)
            setattr(s, edge + field + 'r', 1.)
    # removing the in-plane rigid-body motions
    s.x1u = 0.
    s.y1v = 0.
    s._rebuild()
    return s


def make_callables(model):
    s = make_shell(model)

    def fint(c):
        return np.asarray(s.calc_fint(c=np.ascontiguousarray(c)), dtype=float)

    def KT(c):
        return s.calc_kT(c=np.ascontiguousarray(c)).toarray()

    return s, fint, KT


def random_state(s, rng):
    h_shell = len(s.stack)*s.plyt
    c = h_shell*rng.standard_normal(s.get_size())
    dofs = modelDB.db[s.model]['dofs']
    c[2::dofs] *= 20
    if dofs == 5:
        # independent rotations phix, phiy of the shear deformation theories,
        # of the order of the slopes of w, such that the state is not
        # dominated by the transverse shear strains
        c[3::dofs] *= 20/min(s.a, s.b)
        c[4::dofs] *= 20/min(s.a, s.b)
    return c

In [7]:
# from tests/tests_shell/test_tangent_consistency.py
def test_tangent_is_derivative_of_fint(model, seed):
    s, fint, KT = make_callables(model)
    rng = np.random.default_rng(seed)
    c = random_state(s, rng)
    d = random_state(s, rng)

    f0 = fint(c)
    KTd = KT(c) @ d
    scale = np.linalg.norm(KTd)
    assert scale > 0

    steps = [1.e-2, 1.e-3, 1.e-4, 1.e-5, 1.e-6]
    errors = [np.linalg.norm(fint(c + h*d) - f0 - h*KTd)/(h*scale)
              for h in steps]

    # a consistent tangent leaves a second order remainder, so the error
    # falls by ten for every decade of h; an inconsistent one leaves a first
    # order remainder, so the error plateaus instead
    for h, prev, err in zip(steps[1:], errors[:-1], errors[1:]):
        assert err < 0.2*prev, (
            'error did not fall by ten going to h=%.0e: %.3e -> %.3e; the '
            'tangent is not the derivative of fint' % (h, prev, err))

    # error/h is the constant that multiplies the second derivative, so it
    # must not drift with h
    coefficients = [err/h for h, err in zip(steps, errors)]
    spread = max(coefficients)/min(coefficients)
    assert spread < 2., (
        'error/h drifted by a factor %.1f over %.0e..%.0e, so the remainder '
        'is not second order' % (spread, steps[0], steps[-1]))

    assert errors[-1] < 1.e-4, (
        'residual %.3e at h=%.0e is too large for a consistent tangent'
        % (errors[-1], steps[-1]))

In [8]:
for seed in [0, 1, 2]:
    for model in MODELS:
        test_tangent_is_derivative_of_fint(model=model, seed=seed)
        print(f"test_tangent_is_derivative_of_fint(model={model!r}, seed={seed!r}): passed")

test_tangent_is_derivative_of_fint(model='plate_clpt_donnell', seed=0): passed
test_tangent_is_derivative_of_fint(model='cylshell_clpt_donnell', seed=0): passed
test_tangent_is_derivative_of_fint(model='cylshell_clpt_sanders', seed=0): passed
test_tangent_is_derivative_of_fint(model='plate_fsdt_donnell', seed=0): passed
test_tangent_is_derivative_of_fint(model='plate_tsdt_donnell', seed=0): passed
test_tangent_is_derivative_of_fint(model='cylshell_fsdt_donnell', seed=0): passed
test_tangent_is_derivative_of_fint(model='cylshell_fsdt_sanders', seed=0): passed
test_tangent_is_derivative_of_fint(model='cylshell_tsdt_donnell', seed=0): passed
test_tangent_is_derivative_of_fint(model='cylshell_tsdt_sanders', seed=0): passed
test_tangent_is_derivative_of_fint(model='plate_clpt_gl', seed=0): passed
test_tangent_is_derivative_of_fint(model='plate_clpt_tl', seed=0): passed
test_tangent_is_derivative_of_fint(model='plate_clpt_donnell', seed=1): passed
test_tangent_is_derivative_of_fint(model='cy

test_tangent_is_derivative_of_fint(model='cylshell_clpt_donnell', seed=2): passed
test_tangent_is_derivative_of_fint(model='cylshell_clpt_sanders', seed=2): passed


test_tangent_is_derivative_of_fint(model='plate_fsdt_donnell', seed=2): passed
test_tangent_is_derivative_of_fint(model='plate_tsdt_donnell', seed=2): passed
test_tangent_is_derivative_of_fint(model='cylshell_fsdt_donnell', seed=2): passed
test_tangent_is_derivative_of_fint(model='cylshell_fsdt_sanders', seed=2): passed
test_tangent_is_derivative_of_fint(model='cylshell_tsdt_donnell', seed=2): passed
test_tangent_is_derivative_of_fint(model='cylshell_tsdt_sanders', seed=2): passed
test_tangent_is_derivative_of_fint(model='plate_clpt_gl', seed=2): passed
test_tangent_is_derivative_of_fint(model='plate_clpt_tl', seed=2): passed


The product of the tangent stiffness matrix by a vector, $\boldsymbol{K}_T \boldsymbol{v}$, is
available without assembling $\boldsymbol{K}_T$, at the cost of one evaluation of the
internal force vector, with `Shell.calc_kTv` and
`MultiDomain.calc_kTv`, see [Tangent stiffness matrix and its product by a vector](../tangent_product.rst).

The methods `calc_fext`, `calc_fint`, `calc_kC` and `calc_kG` of
`Shell` and `MultiDomain` have the signatures required by
`structsolve.Analysis`, which implements the Newton-Raphson method with
load control and the arc-length methods, see [Non-linear static analysis with the arc-length methods](ex_arc_length.ipynb).